# Noise Power Spectral Density $S_N(f)$ Modeling

## Overview

The total noise PSD $S_N(f)$ is modeled as the sum of four statistically independent noise contributions:

$$S_{N}(f) = S_{RIN} \cdot |H_{ch}(f)|^2 + S_{shot}(f) + S_{th}(f) + S_{ADC}(f)$$

Each component is detailed below.



In [8]:
import numpy as np
# Physical constants
q = 1.602176634e-19 # Eletron charge (C)
from scipy.constants import k as kB  # Boltzmann constant



### The Necessity of `f_array` and `np.full_like` in PSD Calculations

In numerical signal processing simulations, calculating the total Power Spectral Density (PSD) requires summing both frequency-dependent components (such as filtered RIN) and constant components (such as Thermal and Shot noise).

* **`f_array`**: Defines the continuous frequency domain grid for the simulation. It establishes the mandatory dimension and shape for all spectral arrays in the system.
* **`np.full_like(f_array, constant_value)`**: Converts a scalar noise value into an array that perfectly matches the `f_array` in shape and data type.



## 1. Relative Intensity Noise (RIN)

RIN originates from the transmitter laser. Unlike other noise sources added at the receiver, RIN is added at the transmitter and is consequently filtered by the linear channel's transfer function $H_{ch}(f)$.

The RIN PSD is proportional to the square of the average transmitted optical power $\overline{P_{TX}^2}$:

$$S_{RIN} = k_{RIN} \cdot \overline{P_{TX}^2}$$

Where:

* $k_{RIN} = RIN_{coeff} / 2$ is a proportionality factor.


* $RIN_{coeff}$ is the RIN coefficient expressed in 1/Hz.

* $\overline{P_{TX}^2}$ is the average transmitted optical power squared.

In [9]:
def calc_S_RIN(P_TX_sq_avg, RIN_coeff_dB, fq_array ):
    """
    Calculates the PSD of RIN at the source.
    k_RIN = RIN_coeff_linear / 2.
    Returns a scalar, as the channel filter will be applied in the main function.
    """
    # Convert from dB/Hz to linear scale
    RIN_coeff_lin = 10**(RIN_coeff_dB / 10)
    k_RIN = RIN_coeff_lin / 2
    return np.full_like(fq_array, k_RIN * P_TX_sq_avg)

In [10]:
def calc_S_RIN_opticompy(RIN_var, Fs, fq_array):
    """
    Calculates the PSD of RIN.
    Adapted to exactly match OpticommPy's time-domain generation.
    OpticommPy simply generates white noise with a fixed variance (RIN_var).
    Therefore, the double-sided PSD is the variance divided by Fs.
    
    Parameters:
    - RIN_var: The variance of the RIN noise fed into OpticommPy
    - Fs: Sampling frequency (Hz)
    - fq_array: Frequency array for the PSD (double-sided)
    """
    
    # Calculate the double-sided PSD matching OpticommPy's generation
    S_RIN = RIN_var / Fs
    
    # Return the constant PSD array across all frequencies
    return np.full_like(fq_array, S_RIN)

## 2. Shot Noise

Shot noise is a quantum effect associated with the photodetection process at the receiver. Its variance scales with the received signal power. For this analytical frequency-resolved model, it is approximated using the average received optical power $P_{RX}$.

$$S_{shot}(f) = k_{shot} \cdot P_{RX}$$

Where $k_{shot}$ is a proportionality factor defined as:

$$k_{shot} = G^2 \cdot F \cdot q \cdot R^{-1}$$

* **$G$**: Photodetector gain (for PIN photodiodes, $G = 1$; for APDs, $G > 1$).


* **$F$**: Photodetector excess noise figure (for PIN, $F = 1$).


* **$q$**: Electron charge ($1.602 \times 10^{-19}$ C).


* **$R$**: Photodiode responsivity (A/W).

In [10]:
def calc_S_shot(P_RX, G, F, R, fq_array):
    """
    Calculates the PSD of shot noise.
    Assuming k_shot = (G^2 * F * q )/ R.
    """
    k_shot = (G**2 * F * q) / R
    S_shot = k_shot * P_RX
    return np.full_like(fq_array, S_shot)

In [ ]:
def calc_S_shot_opticompy(P_RX, R, Id, fq_array):
    """
    Calculates the PSD of shot noise.
    Adapted to exactly match OpticommPy's time-domain generation.
    OpticommPy calculates variance as: Fs * q * (ipd + Id)
    Therefore, the double-sided PSD is q * (I_avg + Id).
    
    Note on APD vs PIN Photodiodes:
    The reference article defines the APD shot noise proportionality factor
    as k_shot = (G^2 * F * q) / R. However, the article also explicitly states 
    that for a standard PIN photodiode, G = 1 and F = 1. Since OpticommPy's 
    time-domain implementation does not apply avalanche gain (modeling a PIN), 
    we safely omit G and F to perfectly mirror the simulator's behavior.
    
    Parameters:
    - P_RX: Average received optical power (W)
    - R: Photodiode responsivity (A/W)
    - Id: Dark current (A)
    - fq_array: Frequency array for the PSD (double-sided)
    """
    
    # Calculate the average photocurrent based on received power
    I_avg = P_RX * R
    
    # Calculate the double-sided PSD matching OpticommPy's exact formula
    # S_shot = q * (photocurrent + dark_current)
    # (G and F are inherently 1 here, representing a PIN detector)
    S_shot = q * (I_avg + Id)
    
    #Return the constant PSD array across all frequencies
    return np.full_like(fq_array, S_shot)


## 3. Thermal Noise

Thermal noise is treated as additive white Gaussian noise (AWGN) and is assumed to be constant across the frequency spectrum. It is typically dominated by the internal noise generated by the Transimpedance Amplifier (TIA) at the receiver.

$$S_{th}(f) = \frac{N_0}{2}$$

Where $N_0$ is the equivalent noise power spectral density in $\text{W}^2/\text{Hz}$.

In [11]:
def calc_S_th(N_0, fq_array):
    """
    Calculates the PSD of thermal noise, assumed to be constant across frequency.
    Based on S_th = N_0 / 2.
    """
    return np.full_like(fq_array, N_0 / 2)

In [ ]:
def calc_S_th_opticompy(Tc, RL, fq_array):
    """
    Calculates the PSD of thermal noise.
    Uses OpticommPy parameters to internally calculate N_0,
    then applies the analytical equation S_th = N_0 / 2.
    
    Parameters:
    - Tc: Temperature in Celsius
    - RL: Load Resistance in Ohms
    - fq_array: Frequency array for the PSD (double-sided)
    """
    
    #Convert temperature from Celsius to Kelvin
    T = Tc + 273.15
    
    # Calculate N_0 based on the OpticommPy noise variance definition
    # N_0 = 4 * kB * T / RL
    N_0 = (4 * kB * T) / RL
    
    # Calculate the double-sided PSD (S_th = N_0 / 2)
    S_th = N_0 / 2
    
    #Return the constant PSD array across all frequencies
    return np.full_like(fq_array, S_th)

## 4. ADC Quantization Noise

The analog-to-digital conversion process at the receiver introduces quantization noise, which is dependent on the resolution of the ADC. The PSD of the quantization noise is modeled as:

$$S_{ADC}(f) = \frac{PAPR \cdot \sigma_x^2}{12 \cdot f_s \cdot 2^{(2 \cdot ENOB - 2)}}$$

Where:

* **$PAPR$**: Peak-to-Average Power Ratio of the signal.


* **$ENOB$**: Effective Number of Bits of the oscilloscope/ADC.


* **$f_s$**: Sampling frequency.


*  **$\sigma_x^2$**: Power of the AC-coupled signal after quantization, calculated as $\sigma_x^2 = \int_{-f_s/2}^{f_s/2} S_x(f) df$.

In [1]:
def calc_S_ADC(PAPR, sigma_x_sq, ENOB, fs, fq_array):
    """
    Calculates the PSD of ADC quantization noise.
    Based on the quantization noise model.
    """
    S_ADC = ((PAPR) * sigma_x_sq / (12 * fs) * (2**(2 * ENOB - 2)))
    return np.full_like(fq_array, S_ADC)

In [ ]:
def calc_S_ADC_opticompy(Vmax, Vmin, ENOB, outFs, fq_array):
    """
    Calculates the PSD of ADC quantization noise.
    Uses OpticommPy parameters to derive the exact analytical 
    equation: S_ADC = (PAPR * sigma_x^2) / (12 * fs * 2^(2*ENOB - 2))
    
    Parameters:
    - Vmax: Maximum voltage limit of the ADC clipping
    - Vmin: Minimum voltage limit of the ADC clipping
    - ENOB: Effective Number of Bits
    - outFs: Output sampling frequency of the ADC (Hz) - corresponds to fs
    - fq_array: Frequency array for the PSD (double-sided)
    """
    
    #  Find the peak voltage from the OpticommPy full-scale range
    Vpeak = (Vmax - Vmin) / 2
    
    #  The equation's numerator (PAPR * sigma_x^2) is mathematically equal to Vpeak^2
    PAPR_sigma_x_sq = Vpeak**2
    
    # Apply the exact equation as presented in the article/image
    # S_ADC(f) = (PAPR * sigma_x^2) / (12 * fs * 2^(2*ENOB - 2))
    S_ADC = PAPR_sigma_x_sq / (12 * outFs * (2**(2 * ENOB - 2)))
    
    return np.full_like(fq_array, S_ADC)

# Noise Power Spectral Density $S_N(f)$

$$S_{N}(f) = S_{RIN} \cdot |H_{ch}(f)|^2 + S_{shot}(f) + S_{th}(f) + S_{ADC}(f)$$



In [13]:
def calc_S_N(S_RIN_scalar, S_shot_array, S_th_array, S_ADC_array, H_ch_sq_array):
    """
    Consolidate the total noise S_N(f) at the equalizer input (Eq. 8).
    S_N(f) = S_RIN * |H_ch(f)|^2 + S_shot(f) + S_th(f) + S_ADC(f)
    """

    S_N_array = (S_RIN_scalar * H_ch_sq_array) + S_shot_array + S_th_array + S_ADC_array

    return S_N_array